In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive
from tqdm import tqdm

# ==========================================
# 0. AMBIENTE E SEED
# ==========================================
drive.mount('/content/drive')

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

DATA_PATH = "/content/drive/MyDrive/epPLN/train.xlsx"
TEXT_COL = "resp_text"
TARGET_COL = "clarity"

df = pd.read_excel(DATA_PATH)

In [ ]:
# 1. CLEANING & MAPS
# ==========================================
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r'https?://\S+|www\.\S+', 'LINK_URL', texto)
    texto = re.sub(r'(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./]*', 'LEI_DECRETO', texto)
    texto = re.sub(r'\b\d{5,}\.\d{6}/\d{4}-\d{2}\b', 'NUMERO_PROCESSO', texto)
    texto = re.sub(r'\(\?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}', 'NUMERO_TELEFONE', texto)
    return texto.lower()

df["resp_text_clean"] = df[TEXT_COL].apply(normalizar_texto)

label2id_original = {"c1": 0, "c234": 1, "c5": 2}
id2label_original = {v: k for k, v in label2id_original.items()}
df["label_orig"] = df[TARGET_COL].map(label2id_original)

# Rótulos para o Estágio 1: 0 = Intermediário (c234), 1 = Extremo (c1 ou c5)
df["label_stage1"] = df["label_orig"].apply(lambda x: 0 if x == 1 else 1)

# Rótulos para o Estágio 2 (Apenas c1 vs c5): 0 = c1, 1 = c5
df_stage2 = df[df["label_orig"].isin([0, 2])].copy()
df_stage2["label_stage2"] = df_stage2["label_orig"].apply(lambda x: 0 if x == 0 else 1)

# Split Estratificado Geral (80/20) para avaliação final consistente
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label_orig"])

# Prepara subsets de treino e validação do Estágio 2 (Apenas extremos)
df_train_stage2 = df_train[df_train["label_orig"].isin([0, 2])].copy()
df_train_stage2["label_stage2"] = df_train_stage2["label_orig"].apply(lambda x: 0 if x == 0 else 1)

MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

def encode_texts(texts):
    return tokenizer([str(t) for t in texts], padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="pt")

class TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])


In [ ]:
# 2. FUNÇÃO DE TREINO DE UM CLASSIFICADOR BINÁRIO
# ==========================================
def treinar_modelo_binario(train_texts, train_labels, val_texts, val_labels, nome_modelo, epochs=3):
    print(f"\n==================================================")
    print(f"TREINANDO: {nome_modelo}")
    print(f"==================================================")
    
    train_loader = DataLoader(TextDataset(encode_texts(train_texts), train_labels), batch_size=16, shuffle=True)
    val_loader = DataLoader(TextDataset(encode_texts(val_texts), val_labels), batch_size=16, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
    
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))
    
    best_f1 = 0.0
    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Treino]"):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1)
                preds_all.extend(preds.cpu().numpy())
                labels_all.extend(batch['labels'].numpy())
                
        f1 = f1_score(labels_all, preds_all, average='macro')
        acc = accuracy_score(labels_all, preds_all)
        print(f"--> Época {epoch+1} | Acc: {acc:.4f} | F1-Macro: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), f'{nome_modelo}.pt')
            
    model.load_state_dict(torch.load(f'{nome_modelo}.pt'))
    return model


In [ ]:
# 3. TREINAMENTO DOS DOIS ESTÁGIOS
# ==========================================
# Estágio 1: Intermediário (0) vs Extremos (1)
model_stg1 = treinar_modelo_binario(
    df_train["resp_text_clean"].to_numpy(),
    df_train["label_stage1"].to_numpy(),
    df_val["resp_text_clean"].to_numpy(),
    df_val["label_stage1"].to_numpy(),
    nome_modelo="stage1_filter",
    epochs=3
)

# Estágio 2: c1 (0) vs c5 (1)
model_stg2 = treinar_modelo_binario(
    df_train_stage2["resp_text_clean"].to_numpy(),
    df_train_stage2["label_stage2"].to_numpy(),
    df_val[df_val["label_orig"].isin([0, 2])]["resp_text_clean"].to_numpy(),
    df_val[df_val["label_orig"].isin([0, 2])]["label_orig"].apply(lambda x: 0 if x==0 else 1).to_numpy(),
    nome_modelo="stage2_extremes",
    epochs=3
)

In [ ]:
# 4. AVALIAÇÃO CONJUNTA HIERÁRQUICA
# ==========================================
print("\n==================================================")
print("AVALIANDO PIPELINE MULTI-STAGE NO CONJUNTO DE VALIDAÇÃO")
print("==================================================")

val_loader_full = DataLoader(TextDataset(encode_texts(df_val["resp_text_clean"].to_numpy())), batch_size=16, shuffle=False)

model_stg1.eval()
model_stg2.eval()

preds_stg1 = []
with torch.no_grad():
    for batch in tqdm(val_loader_full, desc="Executando Estágio 1"):
        out = model_stg1(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        preds_stg1.extend(torch.argmax(out.logits, dim=1).cpu().numpy())

preds_stg2 = []
with torch.no_grad():
    for batch in tqdm(val_loader_full, desc="Executando Estágio 2"):
        out = model_stg2(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        preds_stg2.extend(torch.argmax(out.logits, dim=1).cpu().numpy())

# Combinação da Lógica Hierárquica:
# Se Estágio 1 previu 0 -> c234 (id 1)
# Se Estágio 1 previu 1 -> Olha o Estágio 2: Se 0 -> c1 (id 0), Se 1 -> c5 (id 2)
final_preds = []
for p1, p2 in zip(preds_stg1, preds_stg2):
    if p1 == 0:
        final_preds.append(1) # c234
    else:
        final_preds.append(0 if p2 == 0 else 2) # c1 ou c5

y_val_orig = df_val["label_orig"].to_numpy()

acc = accuracy_score(y_val_orig, final_preds)
f1_macro = f1_score(y_val_orig, final_preds, average='macro')

print(f"\n--> Resultado Final Multi-Stage | Val Acc: {acc:.4f} | Val F1-Macro: {f1_macro:.4f}\n")
print("Relatório de Validação Final (Pipeline Multi-Stage):")
print(classification_report(y_val_orig, final_preds, target_names=["c1", "c234", "c5"], digits=4))